In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Master pipeline: run the full workflow end-to-end
# =============================================================================
# Step:         orchestrator (calls Steps 1-7 via papermill)
# Summary:      Run every step notebook in sequence with one set of top-level parameters, producing the AI-ready dataset.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.0.0
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Master pipeline — run the full FUMD-AI preprocessing workflow

Calls `notebooks/step_1_...ipynb` through `notebooks/step_7_...ipynb` in
order via [papermill](https://papermill.readthedocs.io/), passing each
step the output of the previous one. Edit the **parameters cell** below
(or override it headlessly, e.g. `papermill run_pipeline.ipynb out.ipynb
-p RAW_SUMO_XML_PATH my_run.xml -p MAPPING_PATH my_mapping.txt`) and run
this one notebook to go from raw SUMO/OMNeT++ output to the labeled,
AI-ready dataset - no need to open each step notebook individually.

**Run this notebook from the repository root** (same convention as every
individual step notebook), so that `notebooks/...` and `example-data/...`
resolve and Step 6's `sys.path.insert(0, "src")` finds the shared
`fumd_workflow` package.

**What it does, and does not, run by default:**
- Steps 2, 3, 4, 5, 6 always run.
- Step 1's SUMO parsing (`RUN_SUMO_PARSE`) defaults to **on** here - unlike
  the standalone step notebook, whose own default is off - since a full
  pipeline run needs `sumo_trajectory_base.csv` to exist.
- Step 1's OMNeT++ vector extraction (`RUN_OMNET_EXTRACT`) defaults to
  **off**, because it needs the external `extractvectors` tool. With it
  off, Step 3 is fed the `RAW_OMNET_PATH` parameter directly (which
  defaults to the bundled, ready-made `example-data/raw_omnet_export.csv`)
  instead of chaining onto Step 1's output. Set `RUN_OMNET_EXTRACT = True`
  once you have `extractvectors` producing your own `.vec` export, and
  Step 3 automatically switches to reading Step 1's output instead.
- Step 7 (optional diagnostics/plots) defaults to **on**; set
  `RUN_VISUALIZATION = False` to skip it.

**Outputs.** Every intermediate and final file lands in `OUTPUT_DIR`
(default: `pipeline_run/`); a fully executed copy of each step notebook
(with its own outputs, plots and print statements preserved) is written to
`OUTPUT_DIR/executed/` for provenance - open any of those to see exactly
what that step did on this run.


In [ ]:
import os
import papermill as pm


In [ ]:
# ---- Parameters ----
# Defaults point at the tiny bundled example dataset (example-data/) so
# this notebook runs the whole pipeline out of the box - replace with your
# own simulation's paths for a real run.
RAW_SUMO_XML_PATH = "example-data/raw_sumo_fcd.xml"        # Step 1 input (SUMO fcd-output XML)
RAW_OMNET_VEC_PATH = "example-data/raw_omnet_vector.vec"   # Step 1 input (raw OMNeT++ .vec, only used if RUN_OMNET_EXTRACT)
RAW_OMNET_PATH = "example-data/raw_omnet_export.csv"       # Step 3 input, used unless RUN_OMNET_EXTRACT chains onto Step 1 instead
MAPPING_PATH = "example-data/sumo_veins_mapping.txt"       # Step 4 input (SUMO<->VEINS/OMNeT id mapping)

NOTEBOOKS_DIR = "notebooks"    # where the step_*.ipynb notebooks live
OUTPUT_DIR = "pipeline_run"    # every intermediate/final file is written here

RUN_SUMO_PARSE = True     # Step 1, part 1: parse the raw SUMO XML (needed for a full run)
RUN_OMNET_EXTRACT = False # Step 1, part 2: needs the external `extractvectors` tool
RUN_VISUALIZATION = True  # Step 7 (optional diagnostics/plots)

WINDOW_S_VALUES = [3.0]   # Step 6: pre-handover warning window(s), in seconds
TOL_S = 0.1               # Step 6: stable-run tolerance


## Setup

Resolve every input path to an absolute path (so they still work regardless
of `papermill`'s own working directory) and create `OUTPUT_DIR` and its
`executed/` subfolder.


In [ ]:
RAW_SUMO_XML_PATH = os.path.abspath(RAW_SUMO_XML_PATH)
RAW_OMNET_VEC_PATH = os.path.abspath(RAW_OMNET_VEC_PATH)
RAW_OMNET_PATH = os.path.abspath(RAW_OMNET_PATH)
MAPPING_PATH = os.path.abspath(MAPPING_PATH)
NOTEBOOKS_DIR = os.path.abspath(NOTEBOOKS_DIR)
OUTPUT_DIR = os.path.abspath(OUTPUT_DIR)
EXECUTED_DIR = os.path.join(OUTPUT_DIR, "executed")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXECUTED_DIR, exist_ok=True)


def out(filename):
    """Build an absolute path for a pipeline output file inside OUTPUT_DIR."""
    return os.path.join(OUTPUT_DIR, filename)


def run_step(notebook_name, parameters):
    """Execute one step notebook via papermill and print a short summary.

    The executed copy (with its own outputs/plots baked in) is written to
    EXECUTED_DIR, leaving the original notebooks/ files untouched.
    """
    input_path = os.path.join(NOTEBOOKS_DIR, notebook_name)
    output_path = os.path.join(EXECUTED_DIR, notebook_name)
    print(f"=== running {notebook_name} ===")
    pm.execute_notebook(input_path, output_path, parameters=parameters, progress_bar=False)
    print(f"=== finished {notebook_name} -> executed copy: {output_path} ===\n")


print("pipeline outputs:", OUTPUT_DIR)
print("executed notebook copies:", EXECUTED_DIR)


## Step 1 — parse raw SUMO / OMNeT++ output

In [ ]:
run_step("step_1_parse_raw_sumo_and_omnet.ipynb", dict(
    RAW_SUMO_XML_PATH=RAW_SUMO_XML_PATH,
    SUMO_OUTPUT_PATH=out("sumo_trajectory_base.csv"),
    RAW_OMNET_VEC_PATH=RAW_OMNET_VEC_PATH,
    OMNET_OUTPUT_PATH=out("omnet_export.csv"),
    RUN_SUMO_PARSE=RUN_SUMO_PARSE,
    RUN_OMNET_EXTRACT=RUN_OMNET_EXTRACT,
))


## Step 2 — add past-position lag columns

In [ ]:
run_step("step_2_add_past_position_columns.ipynb", dict(
    INPUT_PATH=out("sumo_trajectory_base.csv"),
    OUTPUT_PATH=out("sumo_trajectory.csv"),
))


## Step 3 — generate the OMNeT++ feature matrix

If `RUN_OMNET_EXTRACT` was on, this chains onto Step 1's own output
instead of the `RAW_OMNET_PATH` parameter.


In [ ]:
step3_raw_omnet_path = out("omnet_export.csv") if RUN_OMNET_EXTRACT else RAW_OMNET_PATH

run_step("step_3_generate_omnet_matrix.ipynb", dict(
    RAW_OMNET_PATH=step3_raw_omnet_path,
    CLEAN_OMNET_PATH=out("omnet_clean.csv"),
    OUTPUT_MATRIX_PATH=out("omnet_feature_matrix.csv"),
))


## Step 4 — merge SUMO trajectories with the OMNeT++ feature matrix

In [ ]:
run_step("step_4_merge_sumo_omnet.ipynb", dict(
    VEHICLES_PATH=out("sumo_trajectory.csv"),
    OMNET_PATH=out("omnet_feature_matrix.csv"),
    MAPPING_PATH=MAPPING_PATH,
    COMBINED_PATH=out("combined_dataset.csv"),
))


## Step 5 — fix placeholder past-position values

In [ ]:
run_step("step_5_fix_past_positions.ipynb", dict(
    INPUT_PATH=out("combined_dataset.csv"),
    OUTPUT_PATH=out("combined_dataset_fixed.csv"),
))


## Step 6 — label cell migrations (the AI-ready dataset)

This is the step that produces the final labeled dataset(s), one per
`WINDOW_S_VALUES` entry.


In [ ]:
run_step("step_6_label_cell_migrations.ipynb", dict(
    INPUT_PATH=out("combined_dataset_fixed.csv"),
    WINDOW_S_VALUES=WINDOW_S_VALUES,
    TOL_S=TOL_S,
    TRAJECTORIES_PATH=out("trajectories.csv"),
    OUTPUT_DIR=OUTPUT_DIR,
))

labeled_paths = [out(f"dataset_labeled_w{w:g}.csv") for w in WINDOW_S_VALUES]
print("AI-ready dataset(s):")
for p in labeled_paths:
    print(" -", p)


## Step 7 (optional) — visualize and sanity-check the migration labels

In [ ]:
if RUN_VISUALIZATION:
    run_step("step_7_visualize_migrations_optional.ipynb", dict(
        WINDOW_S_VALUES=WINDOW_S_VALUES,
        TOL_S=TOL_S,
        TRAJECTORIES_PATH=out("trajectories.csv"),
        DATASET_TEMPLATE=out("dataset_labeled_w{w:g}.csv"),
        EVENTS_ALL_TEMPLATE=out("events_all_w{w:g}.csv"),
        EVENT_MAP_WINDOW_S=WINDOW_S_VALUES[0],
    ))
else:
    print("RUN_VISUALIZATION is False - skipping Step 7.")


## Done

The full pipeline has finished. `labeled_paths` above lists the final
AI-ready dataset(s); everything else this run produced (intermediate CSVs,
plus a fully executed copy of every step notebook for provenance) is under
`OUTPUT_DIR`.
